# 03 · End-to-End Evaluation & Risk Tiering

**FraudShield AI** — the full pipeline:

```
User Signup -> Signup Trust Model  -> trust_score
Payment     -> Payment Abuse Model -> payment_risk_score
Admin       -> review flagged accounts -> ban / approve -> new labels
```

In [ ]:
import pandas as pd

from trust_radar.config import FeatureConfig, PaymentModelConfig, SignupGNNConfig
from trust_radar.evaluation.evaluate_signup import evaluate_signup_gnn
from trust_radar.inference.predict_payment import PaymentPredictor
from trust_radar.inference.predict_signup import SignupPredictor
from trust_radar.training.train_payment import train_payment_model
from trust_radar.training.train_signup import train_signup_gnn
from trust_radar.utils.metrics import optimal_threshold_cost
from trust_radar.utils.preprocessing import build_graph_data
from trust_radar.utils.synthetic import (
    synthesize_payment_dataset,
    synthesize_signup_dataset,
    synthesize_signup_edges,
)

cfg = FeatureConfig()

## 1. Stage 1 — Signup Trust Model

In [ ]:
s_df = synthesize_signup_dataset(n=1500, seed=7)
s_data = build_graph_data(
    s_df[cfg.signup_numeric_features],
    synthesize_signup_edges(len(s_df), avg_degree=5.0, seed=7),
    labels=s_df['label'],
)
s_model, _ = train_signup_gnn(s_data, SignupGNNConfig(epochs=25))
s_metrics = evaluate_signup_gnn(s_model, s_data, split='test')
print('signup ROC-AUC :', round(s_metrics['roc_auc'], 4))
print('mean trust     :', round(s_metrics['mean_trust_score'], 1))

In [ ]:
s_pred = SignupPredictor(model_or_path=s_model).predict_graph(s_data)
signup_tiers = pd.Series(s_pred['decision']).value_counts()
print('Signup decision tiers:')
print(signup_tiers)

## 2. Stage 2 — Payment Abuse Model

The upstream `trust_score` is a first-class feature of the payment model.

In [ ]:
p_df = synthesize_payment_dataset(n=6000, seed=7)
X = p_df[cfg.payment_features].copy()
for col in cfg.payment_categorical_features:
    X[col] = X[col].astype('category')

p_model, p_metrics = train_payment_model(
    X, p_df['label'],
    config=PaymentModelConfig(n_estimators=200),
    categorical_features=cfg.payment_categorical_features,
)
print('payment accuracy :', round(p_metrics['accuracy'], 4))
print('macro F1         :', round(p_metrics['macro_f1'], 4))
print('abuse ROC-AUC    :', round(p_metrics['abuse_roc_auc'], 4))

In [ ]:
p_pred = PaymentPredictor(model_or_path=p_model).score_batch(p_df)
print('Payment decision tiers:')
print(p_pred['decision'].value_counts())
print()
print('Predicted abuse types:')
print(p_pred['abuse_type'].value_counts())

## 3. Combined risk-tier summary

In [ ]:
summary = pd.DataFrame({
    'signup': pd.Series(s_pred['decision']).value_counts(),
    'payment': p_pred['decision'].value_counts(),
}).fillna(0).astype(int)
summary

## 4. Cost-optimal operating threshold (payment abuse vs. legit)

Balances false-positive review cost against missed-abuse loss on the binary `label > 0` view driven by the 0-100 `payment_risk_score`.

In [ ]:
risk = p_model.predict_risk(X)
y_bin = (p_df['label'].to_numpy() > 0).astype(int)
best_t, min_cost = optimal_threshold_cost(
    y_bin, risk['payment_risk_score'] / 100.0, cost_fp=5.0, cost_fn=100.0
)
print('optimal threshold:', round(best_t, 3))
print('min total cost   : $', round(min_cost, 2))

## 5. The human-in-the-loop label flywheel

Flagged accounts (`ALLOW_FLAG_REVIEW`, `ALLOW_HIGH_PRIORITY_REVIEW`, `TEMP_SUSPEND_MANUAL_REVIEW`, `BLOCK`) are routed to the admin dashboard. Analyst **ban / approve** verdicts become high-quality training labels (`admin_reviewed`, `review_result`), continuously improving both models.